In [1]:
import torch
import torchaudio

import os

samples_folder = "samples"

# папка семплов собрана руками. имитирует батч
audio_paths = [
    os.path.join(samples_folder, file) 
    for file in os.listdir(samples_folder) 
    if file.endswith(".wav")
]
import os

print(audio_paths)

for path in audio_paths:
    print(f"File {path} exists:", os.path.exists(path))


# все в  1 канал
from pydub import AudioSegment

for path in audio_paths:
    audio = AudioSegment.from_file(path)
    audio = audio.set_channels(1)  
    audio.export(path, format="wav")


import soundfile as sf

for path in audio_paths:
    try:
        with sf.SoundFile(path) as file:
            print(f"{path} has {file.samplerate} Hz sample rate and {file.channels} channels")
    except RuntimeError as e:
        print(f"Error with {path}: {e}")


import soundfile as sf
import torch

def load_audio_with_soundfile(paths, sample_rate=48000):
    audio_batch = []
    for path in paths:
        audio, sr = sf.read(path)
        if sr != sample_rate:
            raise ValueError(f"Sample rate mismatch: {sr} Hz in {path}")
        audio_tensor = torch.tensor(audio).unsqueeze(0) 
        audio_batch.append(audio_tensor)
    return audio_batch

audio_batch = load_audio_with_soundfile(audio_paths)
print("Audio batch loaded successfully using soundfile.")
for i, audio_tensor in enumerate(audio_batch):
    print(f"Audio sample {i} shape: {audio_tensor.shape}")


# Find the maximum length of audio in the batch
max_length = max(audio_tensor.shape[1] for audio_tensor in audio_batch)

# Pad each audio sample to match the max_length and store in padded_audio_batch
padded_audio_batch = [
    torch.nn.functional.pad(audio_tensor, (0, max_length - audio_tensor.shape[1]))
    for audio_tensor in audio_batch
]

# Check shapes after padding
for i, padded_tensor in enumerate(padded_audio_batch):
    print(f"Audio sample {i} padded shape: {padded_tensor.shape}")

# Stack them to create a batch tensor
stacked_batch = torch.stack(padded_audio_batch).float()
print("Stacked Batch Shape:", stacked_batch.shape)


transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=48000, 
    n_fft=1024, 
    hop_length=512, 
    n_mels=128
)

spectrogram_batch = []
for waveform in stacked_batch:
    spectrogram = transform(waveform)  
    spectrogram_batch.append(spectrogram.squeeze(0).T)  

spectrogram_batch = torch.stack(spectrogram_batch)  
print("Spectrogram Batch Shape:", spectrogram_batch.shape)

import pandas as pd

# подтянуть строчки из csv 
dataset_path = "audiofile_dataset_with_merged_features.csv"
df = pd.read_csv(dataset_path)



audio_ids = [file_name.replace(".wav", "") for file_name in audio_paths]
audio_ids = [file_name.replace("samples\\", "") for file_name in audio_paths]

matching_rows = df[df['id'].isin(audio_ids)]

pooled = matching_rows

pooled.to_csv("pooled.csv", index=False)



['samples\\03-01-06-02-01-02-22.wav', 'samples\\03-01-06-02-02-01-22.wav', 'samples\\03-01-06-02-02-02-22.wav', 'samples\\03-01-08-01-01-01-01.wav', 'samples\\03-01-08-01-01-02-01.wav', 'samples\\03-01-08-02-01-01-01.wav']
File samples\03-01-06-02-01-02-22.wav exists: True
File samples\03-01-06-02-02-01-22.wav exists: True
File samples\03-01-06-02-02-02-22.wav exists: True
File samples\03-01-08-01-01-01-01.wav exists: True
File samples\03-01-08-01-01-02-01.wav exists: True
File samples\03-01-08-02-01-01-01.wav exists: True
samples\03-01-06-02-01-02-22.wav has 48000 Hz sample rate and 1 channels
samples\03-01-06-02-02-01-22.wav has 48000 Hz sample rate and 1 channels
samples\03-01-06-02-02-02-22.wav has 48000 Hz sample rate and 1 channels
samples\03-01-08-01-01-01-01.wav has 48000 Hz sample rate and 1 channels
samples\03-01-08-01-01-02-01.wav has 48000 Hz sample rate and 1 channels
samples\03-01-08-02-01-01-01.wav has 48000 Hz sample rate and 1 channels
Audio batch loaded successfully u

c:\Users\pferdlexxie\anaconda3\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)
c:\Users\pferdlexxie\anaconda3\Lib\site-packages\torchaudio\functional\functional.py:584: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (128) may be set too high. Or, the value for `n_freqs` (513) may be set too low.
  warnings.warn(


In [2]:
emotion_mapping = {
    "neutral": 1,
    "calm": 2,
    "happy": 3,
    "sad": 4,
    "angry": 5,
    "fearful": 6,
    "disgust": 7,
    "surprised": 8
}


pooled.loc[:, 'emotion_label'] = pooled['emotion'].map(emotion_mapping)

labels_tensor = torch.tensor(pooled['emotion_label'].values, dtype=torch.long)

print("Labels Tensor Shape:", labels_tensor.shape)
print("Mapped Labels Tensor:", labels_tensor)



Labels Tensor Shape: torch.Size([6])
Mapped Labels Tensor: tensor([6, 6, 6, 8, 8, 8])


C:\Users\pferdlexxie\AppData\Local\Temp\ipykernel_31024\1471947069.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pooled.loc[:, 'emotion_label'] = pooled['emotion'].map(emotion_mapping)


In [3]:
import re
intensity_mapping = {
    "normal": 0,
    "strong": 1,
}

gender_mapping = {
    "male": 0,
    "female": 1
}

pooled.loc[:, 'intensity_encoded'] = pooled['emotional_intensity'].map(intensity_mapping)
pooled.loc[:, 'gender_encoded'] = pooled['gender'].map(gender_mapping)

feature_columns = pooled.drop(columns=['emotion', 'emotion_label', 'emotional_intensity', 'gender', 'vocal_channel', 'id'])
feature_columns['tempo'] = feature_columns['tempo'].apply(lambda x: float(re.sub(r"[\[\]]", "", x)))

additional_features_tensor = torch.tensor(feature_columns.values, dtype=torch.float32)

print("Additional Features Tensor Shape:", additional_features_tensor.shape)



Additional Features Tensor Shape: torch.Size([6, 12])


C:\Users\pferdlexxie\AppData\Local\Temp\ipykernel_31024\1974074177.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pooled.loc[:, 'intensity_encoded'] = pooled['emotional_intensity'].map(intensity_mapping)
C:\Users\pferdlexxie\AppData\Local\Temp\ipykernel_31024\1974074177.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pooled.loc[:, 'gender_encoded'] = pooled['gender'].map(gender_mapping)


In [4]:
print("Column Data Types in feature_columns:\n", feature_columns.dtypes)

non_numeric_columns = feature_columns.select_dtypes(include=['object']).columns
print("Non-numeric columns:", non_numeric_columns)

for column in non_numeric_columns:
    print(f"Unique values in {column}:", feature_columns[column].unique())


Column Data Types in feature_columns:
 repetition                  int64
mfcc_mean                 float64
mfcc_var                  float64
chroma_mean               float64
chroma_var                float64
spectral_contrast_mean    float64
zcr_mean                  float64
rms_mean                  float64
tempo                     float64
duration                  float64
intensity_encoded           int64
gender_encoded              int64
dtype: object
Non-numeric columns: Index([], dtype='object')


### BASELINE MODEL


In [5]:
import torch
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, additional_features_size):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.maxpool = nn.MaxPool1d(2)
        self.hidden_size = hidden_size
        self.additional_features_size = additional_features_size

        self.fc1 = None
        self.fc2 = nn.Linear(hidden_size, output_size)  
    def forward(self, x, additional_features):
        # LSTM layer
        lstm_out, _ = self.lstm(x) 
        
        lstm_out = lstm_out.permute(0, 2, 1)  
        pooled_out = self.maxpool(lstm_out).mean(dim=2) 

        combined_features = torch.cat((pooled_out, additional_features), dim=1)

        if self.fc1 is None:
            fc1_input_size = combined_features.shape[1] 
            self.fc1 = nn.Linear(fc1_input_size, self.hidden_size).to(x.device)

        out = torch.relu(self.fc1(combined_features))
        out = self.fc2(out)
        return out

input_size = spectrogram_batch.shape[2] 
hidden_size = 32 
output_size = 9  
additional_features_size = additional_features_tensor.shape[1] 

model = LSTMModel(input_size, hidden_size, output_size, additional_features_size)

print(model)
print("Input Size:", input_size)


LSTMModel(
  (lstm): LSTM(128, 32, batch_first=True)
  (maxpool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc2): Linear(in_features=32, out_features=9, bias=True)
)
Input Size: 128


In [6]:
import torch
import numpy as np
import random

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)



In [7]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

inp_stacked_batch = spectrogram_batch.to(device)  
additional_features_tensor = additional_features_tensor.to(device)
labels_tensor = labels_tensor.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

num_epochs = 100
for epoch in range(num_epochs):
    model.train() 

    outputs = model(inp_stacked_batch, additional_features_tensor)
    loss = criterion(outputs, labels_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")


model.eval()  
with torch.no_grad(): 
    predictions = model(inp_stacked_batch, additional_features_tensor)
    predicted_classes = torch.argmax(predictions, dim=1)
    print("Predicted Classes:", predicted_classes.cpu().numpy()) 


Epoch [1/100], Loss: 110.2174
Epoch [2/100], Loss: 71.1581
Epoch [3/100], Loss: 36.4872
Epoch [4/100], Loss: 4.1527
Epoch [5/100], Loss: 22.7256
Epoch [6/100], Loss: 36.2059
Epoch [7/100], Loss: 41.0494
Epoch [8/100], Loss: 39.6714
Epoch [9/100], Loss: 33.6061
Epoch [10/100], Loss: 23.8944
Epoch [11/100], Loss: 11.2780
Epoch [12/100], Loss: 4.1843
Epoch [13/100], Loss: 13.9709
Epoch [14/100], Loss: 15.8269
Epoch [15/100], Loss: 11.2867
Epoch [16/100], Loss: 1.5522
Epoch [17/100], Loss: 8.6501
Epoch [18/100], Loss: 14.6634
Epoch [19/100], Loss: 16.8557
Epoch [20/100], Loss: 15.6968
Epoch [21/100], Loss: 11.5894
Epoch [22/100], Loss: 4.8788
Epoch [23/100], Loss: 4.5032
Epoch [24/100], Loss: 9.8104
Epoch [25/100], Loss: 9.4021
Epoch [26/100], Loss: 4.0278
Epoch [27/100], Loss: 3.4649
Epoch [28/100], Loss: 7.2140
Epoch [29/100], Loss: 7.7751
Epoch [30/100], Loss: 5.4781
Epoch [31/100], Loss: 0.7985
Epoch [32/100], Loss: 6.3553
Epoch [33/100], Loss: 8.9362
Epoch [34/100], Loss: 6.3947
Epoch

better models - next commit
